In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence

torch.manual_seed(0)

# Simulate encoder output for a batch of 4 samples
mu    = torch.tensor([0.5, -1.0, 0.2, 0.8])
logvar = torch.tensor([-0.5, 0.0, -1.0, 0.3])
sigma  = torch.exp(0.5 * logvar)

# Reparameterize
eps = torch.randn_like(mu)
z   = mu + sigma * eps

# --- KL divergence (closed form) ---
kl_closed = -0.5 * torch.sum(1 + logvar - mu**2 - torch.exp(logvar))

# --- KL divergence (via torch.distributions, for verification) ---
q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(mu))
kl_lib = torch.sum(kl_divergence(q, p))

print(f"KL (closed form): {kl_closed:.4f}")
print(f"KL (torch lib):   {kl_lib:.4f}")
# These should match (within floating point error)

# --- Fake reconstruction loss (MSE as a stand-in) ---
x_original    = torch.randn(4, 8)
x_reconstructed = torch.randn(4, 8)   # replace with model output in the assignment
recon_loss = F.mse_loss(x_reconstructed, x_original, reduction='sum')

elbo = recon_loss + kl_closed
print(f"\nReconstruction loss: {recon_loss:.4f}")
print(f"ELBO (loss):         {elbo:.4f}")

KL (closed form): 1.2271
KL (torch lib):   1.2271

Reconstruction loss: 65.3049
ELBO (loss):         66.5320


Part 1

In [ ]:
def kl_divergence_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """
    Closed-form KL divergence between N(mu, exp(logvar)) and N(0, 1).
    Returns a scalar (summed over all dimensions and batch).
    """
    # YOUR CODE HERE
    kl_closed= -0.5*torch.sum(1+logvar-mu**2-torch.exp(logvar))
    return kl_closed

# Test it
mu     = torch.randn(32, 16)   # batch of 32, latent dim 16
logvar = torch.randn(32, 16)
sigma  = torch.exp(0.5 * logvar)

your_kl = kl_divergence_gaussian(mu, logvar)

q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(sigma))
lib_kl  = kl_divergence(q, p).sum()

print(f"Your KL:   {your_kl:.4f}")
print(f"Torch KL:  {lib_kl:.4f}")
assert torch.isclose(your_kl, lib_kl, atol=1e-4), "KL values don't match!"
print("✅ Part 1 passed")

Your KL:   432.7279
Torch KL:  432.7278
✅ Part 1 passed


Part 2

In [ ]:
def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """
    Sample z using the reparameterization trick.
    z = mu + std * eps, where eps ~ N(0, I)
    """
    # YOUR CODE HERE
    return mu+(torch.exp(0.5*logvar))*(torch.randn_like(mu))

# Verify: the mean of many samples should be close to mu
mu_test     = torch.tensor([2.0, -1.0])
logvar_test = torch.tensor([0.0,  0.5])

samples = torch.stack([reparameterize(mu_test, logvar_test) for _ in range(10000)])
print(f"Target mu:       {mu_test.tolist()}")
print(f"Sample mean:     {samples.mean(0).tolist()}")
print(f"Target std:      {torch.exp(0.5 * logvar_test).tolist()}")
print(f"Sample std:      {samples.std(0).tolist()}")
# Means and stds should be close (within ~0.05)
print("✅ Part 2 passed")

Target mu:       [2.0, -1.0]
Sample mean:     [2.0181758403778076, -1.008439540863037]
Target std:      [1.0, 1.2840254306793213]
Sample std:      [1.0103050470352173, 1.282791256904602]
✅ Part 2 passed


Part 3

In [ ]:
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor) -> torch.Tensor:
    """
    ELBO loss = Reconstruction loss + KL divergence
    - Reconstruction: BCE between x_recon and x, summed over pixels and batch
    - KL: closed-form KL between N(mu, exp(logvar)) and N(0, I)
    Returns a scalar.
    """
    # YOUR CODE HERe
    recon_loss=F.binary_cross_entropy(x_recon,x,reduction="sum")
    kl_loss=-0.5*torch.sum(1+logvar-mu**2-torch.exp(logvar))
    return recon_loss+kl_loss

# Quick smoke test
batch, dim, latent = 16, 784, 32
x      = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu     = torch.randn(batch, latent)
logvar = torch.randn(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")
assert loss.ndim == 0, "Loss must be a scalar!"
print("✅ Part 3 passed")

ELBO loss (should be a positive scalar): 10559.0469
✅ Part 3 passed


1. We cannot maximize $\log p(x)$ directly because the integral $\int p(z|x) p(z) dz$ is intractable (cannot be calculated) because there is no analytical solution and numerical methods for approximation will fail when there are large number of latent variables $z$. Since direct integration is not possible so we use bayes' theorem to say
$$ p(z|x)=\frac{p(z)p(x|z)}{p(x)}$$ so,
$$ p(x)=\frac{p(z,x)}{p(z|x)}$$ The posterior($p(z|x)$) here is still intractable because the marginal likelihood($p(x)$) is intractable so to approximate it we use the encoder distribution so that we can estimate it using the tractable encoder function and then maximize the RHS to maximize the LHS.
2. The KL term pushes the encoder to the distribution of $z$ which is the unit normal gaussian. If this is not done then the first term alone makes the $z$ distribution go anywhere and leave the unit gaussian we want. So, when we generate new data by the decoder it can happen that during training it saw data very far from the unit gaussian and now when $z$ would be sampled from the unit gaussian since it did not have any data there it would generate garbage.
3. The sampling operation to get a $z$ from the distribution is not differentiable so we cannot gradient descent backwards through this node. The reparametrization trick solves this by using the $\mu$ and $\sigma$ which are calculated deterministically so gradients flow through them backwards during training.
4. When $\mu$ and $\sigma$ are $0$ then the KL term is $0$ which does match the closed form formula. It should be $0$ because then the distribution $q$ is also a normal distribution with mean $0$ and variance $1$ which means its same as $p$ so KL loss should be $0$.So the ELBO loss is just the first term, the reconstruction term.